In [ ]:
from google.colab import drive

drive.mount('/content/drive')


# from google.colab import drive

# drive.mount('/content/drive', force_remount=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# craetion de path

from pathlib import Path

# Dossier principal
PROJECT_ROOT = Path("/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR")

# Structure du projet
folders = [
    "data/raw",
    "data/processed",
    "data/indexes/bm25",
    "data/indexes/vector",
    "data/indexes/hybrid",
    "notebooks",
    "src/retrieval",
    "src/ingestion",
    "tests",
    "results",
    "reports"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully!")
print("Project location:", PROJECT_ROOT)


Project folders created successfully!
Project location: /content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR


In [25]:
# verifier zip file dans drive

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

zip_files = list(RAW_DIR.glob("*.zip"))

print("Nombre de fichiers ZIP :", len(zip_files))

for file in zip_files:
    print(f"📦 {file.name}")
    print(f"   Taille : {file.stat().st_size / (1024 * 1024):.2f} MB")


Nombre de fichiers ZIP : 2
📦 embeddings_ocr.zip
   Taille : 3.97 MB
📦 embeddings_fstbm.zip
   Taille : 0.18 MB


In [26]:
#tester le continue des ZIP


import zipfile

for zip_path in zip_files:
    print("\n" + "=" * 65)
    print("ARCHIVE :", zip_path.name)
    print("=" * 65)

    with zipfile.ZipFile(zip_path, "r") as archive:
        files = archive.infolist()

        print("Nombre total d'éléments :", len(files))

        for item in files[:100]:
            if item.is_dir():
                print("📁", item.filename)
            else:
                size_kb = item.file_size / 1024
                print(f"📄 {item.filename} ({size_kb:.1f} KB)")

        if len(files) > 100:
            print("... Autres fichiers non affichés")



ARCHIVE : embeddings_ocr.zip
Nombre total d'éléments : 5
📁 embeddings_ocr/
📄 embeddings_ocr/all_chunks.jsonl (11787.1 KB)
📄 embeddings_ocr/embeddings.npy (3924.1 KB)
📄 embeddings_ocr/embedding_ids.jsonl (112.0 KB)
📄 embeddings_ocr/embedding_manifest.json (0.7 KB)

ARCHIVE : embeddings_fstbm.zip
Nombre total d'éléments : 5
📁 embeddings_fstbm/
📄 embeddings_fstbm/all_chunks.jsonl (159.6 KB)
📄 embeddings_fstbm/embeddings.npy (176.1 KB)
📄 embeddings_fstbm/embedding_ids.jsonl (5.0 KB)
📄 embeddings_fstbm/embedding_manifest.json (0.7 KB)


In [ ]:
#verifier les file deep one   décompresse les 2 ZIP dans data/embeddings/ \\ lit le manifeste de chaque corpus.  \\

import zipfile, json
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
EMB_DIR = PROJECT_ROOT / "data" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

# 1. Extraire (sans écraser ce qui existe déjà)
for zip_path in RAW_DIR.glob("embeddings_*.zip"):
    target = EMB_DIR / zip_path.stem
    if target.exists():
        print("Déjà extrait :", zip_path.stem)
        continue
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(EMB_DIR)
    print("Extrait :", zip_path.stem)

# 2 et 3. Vérifier chaque corpus
for name in ("embeddings_fstbm", "embeddings_ocr"):
    folder = EMB_DIR / name
    manifest = json.loads((folder / "embedding_manifest.json").read_text(encoding="utf-8"))
    n_chunks = sum(1 for line in (folder / "all_chunks.jsonl").open(encoding="utf-8") if line.strip())
    n_ids = sum(1 for line in (folder / "embedding_ids.jsonl").open(encoding="utf-8") if line.strip())
    vectors = np.load(folder / "embeddings.npy")

    print("\n" + "=" * 50)
    print(name)
    print("Modèle        :", manifest["model_name"])
    print("Chunks        :", n_chunks)
    print("Ids           :", n_ids)
    print("Vecteurs      :", vectors.shape)
    print("Normalisés    :", bool(np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)))
    print("Tout cohérent :", n_chunks == n_ids == vectors.shape[0] == manifest["count"])


    #YES TERE R GOOD TO WORK WITH NOW AND THEY EXISTE ON ENBEDDING FOLDER

Déjà extrait : embeddings_fstbm
Déjà extrait : embeddings_ocr

embeddings_fstbm
Modèle        : BAAI/bge-m3
Chunks        : 44
Ids           : 44
Vecteurs      : (44, 1024)
Normalisés    : True
Tout cohérent : True

embeddings_ocr
Modèle        : BAAI/bge-m3
Chunks        : 981
Ids           : 981
Vecteurs      : (981, 1024)
Normalisés    : True
Tout cohérent : True


In [ ]:
from pathlib import Path
root = Path("/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR")
for folder in ("src", "src/retrieval", "tests", "scripts"):
    (root / folder).mkdir(parents=True, exist_ok=True)
    (root / folder / "__init__.py").touch()
print(sorted(str(p.relative_to(root)) for p in root.rglob("*.py")))

['scripts/.py', 'scripts/__init__.py', 'scripts/demo_search.py', 'src/__init__.py', 'src/retrieval/.py', 'src/retrieval/__init__.py', 'src/retrieval/bm25.py', 'src/retrieval/config.py', 'src/retrieval/hybrid.py', 'src/retrieval/loader.py', 'src/retrieval/models.py', 'src/retrieval/query_encoder.py', 'src/retrieval/search.py', 'src/retrieval/vector_store.py', 'tests/__init__.py', 'tests/helpers.py', 'tests/test_bm25.py', 'tests/test_filters.py', 'tests/test_hybrid.py', 'tests/test_loader.py', 'tests/test_roles.py', 'tests/test_vector.py']


In [5]:
%cd /content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR
!pip -q install chromadb sentence-transformers
!python -m scripts.check_bm25

/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR


In [6]:
%cd /content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR
from pathlib import Path

attendus = [
    "data/embeddings/embeddings_fstbm/all_chunks.jsonl",
    "data/embeddings/embeddings_fstbm/embeddings.npy",
    "data/embeddings/embeddings_fstbm/embedding_ids.jsonl",
    "data/embeddings/embeddings_ocr/all_chunks.jsonl",
    "data/embeddings/embeddings_ocr/embeddings.npy",
    "data/embeddings/embeddings_ocr/embedding_ids.jsonl",
    "src/__init__.py", "src/retrieval/__init__.py",
    "src/retrieval/config.py", "src/retrieval/loader.py", "src/retrieval/models.py",
    "src/retrieval/vector_store.py", "src/retrieval/bm25.py", "src/retrieval/hybrid.py",
    "src/retrieval/query_encoder.py", "src/retrieval/search.py",
    "scripts/__init__.py", "scripts/check_utils.py", "scripts/check_bm25.py",
]
for f in attendus:
    print("OK     " if Path(f).exists() else "MANQUE ", f)

/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR
OK      data/embeddings/embeddings_fstbm/all_chunks.jsonl
OK      data/embeddings/embeddings_fstbm/embeddings.npy
OK      data/embeddings/embeddings_fstbm/embedding_ids.jsonl
OK      data/embeddings/embeddings_ocr/all_chunks.jsonl
OK      data/embeddings/embeddings_ocr/embeddings.npy
OK      data/embeddings/embeddings_ocr/embedding_ids.jsonl
OK      src/__init__.py
OK      src/retrieval/__init__.py
OK      src/retrieval/config.py
OK      src/retrieval/loader.py
OK      src/retrieval/models.py
OK      src/retrieval/vector_store.py
OK      src/retrieval/bm25.py
OK      src/retrieval/hybrid.py
OK      src/retrieval/query_encoder.py
OK      src/retrieval/search.py
OK      scripts/__init__.py
OK      scripts/check_utils.py
OK      scripts/check_bm25.py


In [16]:
!python -m pytest tests/test_bm25.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 4 items                                                              

tests/test_bm25.py::test_tokenize_removes_accents_and_stopwords PASSED   [ 25%]
tests/test_bm25.py::test_bm25_finds_real_article PASSED                  [ 50%]
tests/test_bm25.py::test_bm25_works_on_ocr_corpus PASSED                 [ 75%]
tests/test_bm25.py::test_bm25_unknown_word_returns_nothing PASSED        [100%]

============================== 4 passed in 3.08s ===============================


In [17]:
!python -m pytest tests/test_hybrid.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 4 items                                                              

tests/test_hybrid.py::test_rrf_prefers_chunks_found_by_both PASSED       [ 25%]
tests/test_hybrid.py::test_three_modes_return_the_target PASSED          [ 50%]
tests/test_hybrid.py::test_hybrid_with_role_and_document_type PASSED     [ 75%]
tests/test_hybrid.py::test_invalid_inputs PASSED                         [100%]

============================== 4 passed in 3.67s ===============================


In [15]:
!pip install pytest
!python -m pytest tests/test_vector.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 3 items                                                              

tests/test_vector.py::test_where_clause_role_only PASSED                 [ 33%]
tests/test_vector.py::test_where_clause_role_and_document_type PASSED    [ 66%]
tests/test_vector.py::test_metadata_is_chroma_compatible PASSED          [100%]

============================== 3 passed in 1.98s ===============================


In [7]:
!python -m scripts.check_bm25

--- FST corpus ---
      19.3701 | p.6 | Article 5.1                    | "Article 5.1\nL'inscription pédagogique à un semestre (automne"
      5.6762 | p.3 | Article 1.3 :                  | 'Article 1.3 :\no Le calendrier des examens de fin de semestre'
      5.6066 | p.None | LEGAL_CONTENT                  | "Comment s'inscrire sur la plateforme E-Scolarité?\nAccédez à "
[PASS] FST: Article 5.1 in top 5
[PASS] FST: all results come from fstbm
--- OCR corpus ---
      6.7162 | p.33 | Rabat, le 21 Kaada 1417 (31 ma | 'Rabat, le 21 Kaada 1417 (31 mars 1997).\nHASSAN ABOU AYOUB.\n\n'
      6.6103 | p.2 | TABLE_OF_CONTENTS              | 'marocaine des retraites...\n\nRabat, Casablanca, Fès, Marrakec'
      6.3390 | p.9 | Dahir du 3 août 1955 (14 hija  | 'Dahir du 3 août 1955 (14 hija 1374) ES 7 7\nportant nominatlo'
[PASS] OCR: results found
[PASS] OCR: all are bulletin_officiel
--- roles ---
[PASS] admin can search
[PASS] role None rejected
[PASS] role 'hacker' rejected
[PASS] role 'le

In [8]:
!python -m scripts.check_vector

Indexed in Chroma: {'fstbm': 44, 'ocr': 981}
[PASS] FST indexed: 44
[PASS] OCR indexed: 981
modules.json: 100% 349/349 [00:00<00:00, 900kB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 364kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 22.1MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 173kB/s]
config.json: 100% 687/687 [00:00<00:00, 2.21MB/s]

pytorch_model.bin: downloading bytes:   6% 128M/2.27G [00:01<00:22, 95.7MB/s, 11.0MB/s  ]
pytorch_model.bin: downloading bytes:  13% 289M/2.27G [00:03<00:24, 82.4MB/s, 24.0MB/s  ]
pytorch_model.bin: downloading bytes:  21% 466M/2.27G [00:04<00:16, 113MB/s, 36.1MB/s  ]
pytorch_model.bin: downloading bytes:  23% 514M/2.27G [00:04<00:15, 111MB/s, 39.3MB/s  ]
pytorch_model.bin: downloading bytes:  25% 565M/2.27G [00:05<00:11, 144MB/s, 41.8MB/s  ]
pytorch_model.bin: downloading bytes:  33% 752M/2.27G [00:06<00:06, 229MB/s, 53.8MB/s  ]
pytorch_model.bin: downloading bytes:  37% 835M/2.27G [00:06<00:09, 158MB/s, 58.6MB

In [9]:
!python -m scripts.check_all

[PASS] both corpora indexed  -> {'fstbm': 44, 'ocr': 981}
Loading weights: 100% 391/391 [00:00<00:00, 10493.61it/s]
[PASS] fstbm | vector | admin : results from the right corpus  -> 5 results
[PASS] fstbm | vector | reader: results from the right corpus  -> 5 results
[PASS] fstbm | bm25   | admin : results from the right corpus  -> 5 results
[PASS] fstbm | bm25   | reader: results from the right corpus  -> 5 results
[PASS] fstbm | hybrid | admin : results from the right corpus  -> 5 results
[PASS] fstbm | hybrid | reader: results from the right corpus  -> 5 results
[PASS] ocr | vector | admin : results from the right corpus  -> 5 results
[PASS] ocr | vector | reader: results from the right corpus  -> 5 results
[PASS] ocr | bm25   | admin : results from the right corpus  -> 5 results
[PASS] ocr | bm25   | reader: results from the right corpus  -> 5 results
[PASS] ocr | hybrid | admin : results from the right corpus  -> 5 results
[PASS] ocr | hybrid | reader: results from the right corpu

In [19]:
!python -m scripts.check_hybrid

--- FST | vector ---
Loading weights: 100% 391/391 [00:00<00:00, 12852.15it/s]
      0.8114 | p.6 | Article 5.1                    | "Article 5.1\nL'inscription pédagogique à un semestre (automne"
      0.6328 | p.3 | Article 1.6 :                  | 'Article 1.6 :\no Toute absence à un examen planifié sera sanc'
      0.5779 | p.5 | Article 3.2                    | 'Article 3.2\nLes sanctions appliquées sont graduelles et vari'
[PASS] FST vector: Article 5.1 in top 5
[PASS] FST vector: method label is 'vector'
--- FST | bm25 ---
      24.2074 | p.6 | Article 5.1                    | "Article 5.1\nL'inscription pédagogique à un semestre (automne"
      5.6762 | p.3 | Article 1.3 :                  | 'Article 1.3 :\no Le calendrier des examens de fin de semestre'
      5.6066 | p.None | LEGAL_CONTENT                  | "Comment s'inscrire sur la plateforme E-Scolarité?\nAccédez à "
[PASS] FST bm25: Article 5.1 in top 5
[PASS] FST bm25: method label is 'bm25'
--- FST | hybrid ---
      0

In [13]:
!python -m scripts.test_bm25_interactive

===== LEGAL RAG - BM25 SEARCH =====

Enter your question: la note de rattrapage pour les master
Choose corpus (fstbm / ocr): fstbm
Enter your role (admin / reader): reader
Number of results (Top K): 2

===== SEARCH RESULTS =====

Result 1
Score: 7.4534
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 1. 7 :
Page: 4
Text: Article 1. 7 :
L'étudiant n'ayant pas validé un module et ayant obtenu une note strictement inférieure
à 10 (0:,;note<10), est autorisé à passer un contrôle de rattrapage selon les modalités
suivantes:
□
Pour toute inscription, l'étudiant n'a droit qu'à un seul rattrapage par module;
o
Les contrôles de rattrapage sont organisés à la fin du semestre, durant lequel les
modules concernés sont programmés ;
o
L'étudiant doit conserver pour le rattrapage les notes obtenues dans les éléments
du module 

Result 2
Score: 3.2923
Title: Futurs étudiants
Section: LEGAL_CONTENT
Page: None
Text: Futurs étudiants
La Faculté des Scien

In [18]:
!python -m scripts.test_vector_interactive.py

===== LEGAL RAG - VECTOR SEARCH =====

Indexed collections: {'fstbm': 44, 'ocr': 981}

Enter your question: Comment un étudiant absent peut-il effectuer son inscription pédagogique au semestre suivant ?
Choose corpus (fstbm / ocr): fstbm
Enter your role (admin / reader): reader
Number of results (Top K): 3
5
3
Loading weights: 100% 391/391 [00:00<00:00, 9872.22it/s]

Question encoded successfully!
Vector dimensions: 1024

===== VECTOR SEARCH RESULTS =====

Result 1
Score: 0.7305
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 5.1
Page: 6
Document type: normes_pedagogiques
Text: Article 5.1
L'inscription pédagogique à un semestre (automne ou printemps) est suspendue pour
tout étudiant absent au semestre précédent, sauf demande avec justificatifs d'absence.

Result 2
Score: 0.6291
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 1.6 :
Page: 3
Document type: normes_pedagogiques
Text: Art

In [20]:
!python -m scripts.test_hybrid_interactive

===== LEGAL RAG - HYBRID SEARCH =====

Enter your question: le note de rattrapage
Choose corpus (fstbm / ocr): fstbm
Enter your role (admin / reader): reader

========== VECTOR ==========
Loading weights: 100% 391/391 [00:00<00:00, 13741.22it/s]

Result 1
Score: 0.6086
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 1. 7 :
Page: 4
Document type: normes_pedagogiques
Text: Article 1. 7 :
L'étudiant n'ayant pas validé un module et ayant obtenu une note strictement inférieure
à 10 (0:,;note<10), est autorisé à passer un contrôle de rattrapage selon les modalités
suivantes:
□
Pour toute inscription, l'étudiant n'a droit qu'à un seul rattrapage par module;
o
Les contrôles de rattrapage sont organisés à la fin du semestre, durant lequel les
modules concernés sont programmés ;
o
L'étudiant doit conserver pour le rattrapage les notes obtenues dans les éléments
du module 

Result 2
Score: 0.5307
Title: Normes pédagogiques complémentaires de la 

Re_Ranking

In [21]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 10.5 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [23]:
!python -m scripts.check_reranker
#TEST JUSTE POUR HYBRID

config.json: 100% 795/795 [00:00<00:00, 2.22MB/s]

model.safetensors: downloading bytes:  15% 344M/2.27G [00:03<00:17, 112MB/s, 28.0MB/s  ]
model.safetensors: downloading bytes:  16% 359M/2.27G [00:03<00:20, 95.5MB/s, 29.3MB/s  ]
model.safetensors: downloading bytes:  23% 531M/2.27G [00:04<00:12, 141MB/s, 41.7MB/s  ]
model.safetensors: downloading bytes:  39% 896M/2.27G [00:07<00:15, 90.5MB/s, 60.5MB/s  ]
model.safetensors: downloading bytes:  53% 1.21G/2.27G [00:12<00:11, 89.1MB/s, 60.4MB/s  ]
model.safetensors: downloading bytes:  67% 1.51G/2.27G [00:17<00:03, 212MB/s, 63.5MB/s  ]
model.safetensors: downloading bytes:  67% 1.52G/2.27G [00:29<00:03, 212MB/s, 65.2MB/s  ]
model.safetensors: reconstructing file:  80% 1.81G/2.27G [00:33<00:10, 43.8MB/s, 75.5MB/s  ]
model.safetensors: reconstructing file:  80% 1.81G/2.27G [00:49<00:10, 43.8MB/s, 52.8MB/s  ]
model.safetensors: downloading bytes: 100% 1.52G/1.52G [00:56<00:00, 27.0MB/s, 65.2MB/s  ]
model.safetensors: reconstructing file: 100

In [24]:
# test the whole piplne now
!python -m scripts.check_pipeline

Loading weights: 100% 393/393 [00:00<00:00, 707.47it/s]
Enter your question: la note de rattrapage
Choose corpus (fstbm / ocr): fstbm
Enter role (admin / reader): reader
Loading weights: 100% 391/391 [00:00<00:00, 9220.01it/s]

===== FINAL LEGAL SEARCH RESULTS =====

Result 1
Score: 0.9814
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 1. 7 :
Page: 4
Text: Article 1. 7 :
L'étudiant n'ayant pas validé un module et ayant obtenu une note strictement inférieure
à 10 (0:,;note<10), est autorisé à passer un contrôle de rattrapage selon les modalités
suivantes:
□
Pour toute inscription, l'étudiant n'a droit qu'à un seul rattrapage par module;
o
Les contrôles de rattrapage sont organisés à la fin du semestre, durant lequel les
modules concernés sont programmés ;
o
L'étudiant doit conserver pour le rattrapage les notes obtenues dans les éléments
du module 

Result 2
Score: 0.3907
Title: Normes pédagogiques complémentaires de la licence en sci

In [27]:
!python -m scripts.check_reranking

Loading weights: 100% 393/393 [00:00<00:00, 1064.40it/s]

--- fstbm | vector + rerank ---
Loading weights: 100% 391/391 [00:00<00:00, 8773.80it/s]
  before rerank:
   1. score=0.8114 | p.6 | Article 5.1                  | "Article 5.1\nL'inscription pédagogique à un semestre (automne"
   2. score=0.6328 | p.3 | Article 1.6 :                | 'Article 1.6 :\no Toute absence à un examen planifié sera sanc'
   3. score=0.5779 | p.5 | Article 3.2                  | 'Article 3.2\nLes sanctions appliquées sont graduelles et vari'
   4. score=0.5601 | p.6 | Article 5.3                  | "Article 5.3\no\nL'étudiant n'ayant pas validé sa première anné"
   5. score=0.5287 | p.4 | Article 2.2                  | "Article 2.2\no\nL'étudiant ne conserve que le matériel autoris"
  after rerank:
   1. score=0.9987 | p.6 | Article 5.1                  | "Article 5.1\nL'inscription pédagogique à un semestre (automne"
   2. score=0.1378 | p.6 | Article 5.3                  | "Article 5.3\no\nL'étudiant 

In [28]:
%run -m scripts.run_pipeline

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Enter your question: inscription pédagogique suspendue pour un étudiant absent
Choose corpus (fstbm / ocr): fstbm
Enter role (admin / reader): reader
Choose search type (vector / bm25 / hybrid): vector
Document type, or Enter for all (charte / guide / normes_pedagogiques / page_information): normes_pedagogiques 


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


===== FINAL RESULTS (vector + rerank) =====

Result 1
Score: 0.997
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 5.1
Page: 6
Text: Article 5.1
L'inscription pédagogique à un semestre (automne ou printemps) est suspendue pour
tout étudiant absent au semestre précédent, sauf demande avec justificatifs d'absence.

Result 2
Score: 0.1439
Title: Normes pédagogiques complémentaires de la licence en sciences et techniques
Section: Article 1.6 :
Page: 3
Text: Article 1.6 :
o Toute absence à un examen planifié sera sanctionnée par l'attribution de
l'observation "absent (ABS)"'. L'absence doit être justifiée par un document remis
au service des affaires estudiantines dans un délai de 72 h.
o Si l'absence à un contrôle final de la session ordinaire est justifiée dans les 72 h qui
suivent, l'étudiant est autorisé à passer le contrôle de rattrapage.
o L'absence à un seul TP, justifiée dans les 72 h suivant celle-ci, permet à l'étu

si cela est 

In [29]:
from pathlib import Path
for p in Path("/content/drive/MyDrive").glob("Legal_RAG_Project_AMEZOUAR*"):
    print(p, "| has src/retrieval:", (p / "src" / "retrieval").exists())

/content/drive/MyDrive/Legal_RAG_Project_AMEZOUAR | has src/retrieval: True
